# Estudo guiado — detecção de defeito por banco de embeddings

Notebook de **preencher lacunas**. Cada etapa tem uma célula com `raise NotImplementedError("sua vez!")` — você a substitui pela sua implementação — seguida de uma célula de **check** que importa a referência do pacote `image_anomaly_lab` (o gabarito comentado em `src/`) e confere o seu resultado.

As Partes 1–4 são o **núcleo algorítmico** e rodam só com `numpy`/`scikit-learn` (sem torch, sem baixar dados) — dá para aprender o essencial agora. A Parte 5 junta tudo em imagens reais e precisa do torch (ROCm) e do MVTec baixado.

Rode assim (a partir da raiz do projeto):
```bash
pip install -e '.[dev]'  # núcleo
pip install jupyter
jupyter lab notebooks/estudo_deteccao_defeito.ipynb
```

Antes de olhar o gabarito, leia a docstring do módulo citado em cada parte e **tente prever** o que sua função deve devolver.

In [ ]:
import numpy as np

# Uma nuvem de embeddings 'normais' (aperto em torno de um centro) para brincar
# sem depender de imagens. Cada linha e' o embedding de um 'patch'.
rng = np.random.default_rng(0)

def normal_cluster(n, dim=16, center=0.0, spread=0.05):
    return (rng.standard_normal((n, dim)) * spread + center).astype(np.float32)

good_patches = normal_cluster(2000)          # 'imagens boas' -> banco
test_inliers = normal_cluster(50)            # patches normais de teste
test_outliers = normal_cluster(50, center=5.0)  # patches 'defeituosos' (longe)
print(good_patches.shape, test_inliers.shape, test_outliers.shape)

## Parte 1 — De mapa de features a vetores de patch

*Leia: `to_patch_vectors` em `src/image_anomaly_lab/backbones.py`.*

O backbone devolve um mapa `(B, C, H, W)` — um vetor de `C` canais por posição
espacial (patch). O banco de memória e' um conjunto **plano** de vetores `(N, C)`.
Sua tarefa: achatar `(B, C, H, W)` em `((B*H*W), C)` **sem misturar canais com
posicoes**. Dica: `permute`/`transpose` para `(B, H, W, C)` *antes* do reshape.

In [ ]:
def to_patch_vectors(feature_map):
    """(B, C, H, W) -> ((B*H*W), C). Aqui uso numpy; no src e' o mesmo em torch."""
    # SUA VEZ: mova o eixo de canais para o fim e depois faca o reshape.
    raise NotImplementedError("sua vez! veja a dica acima")

In [ ]:
# CHECK — valores conhecidos (sem torch). Cada patch deve juntar os 3 canais
# da MESMA posicao. Se canais e posicoes se misturarem, estas linhas mudam.
fake = np.arange(2*3*2*2, dtype=np.float32).reshape(2, 3, 2, 2)  # B=2,C=3,H=W=2
mine = to_patch_vectors(fake)
assert mine.shape == (8, 3), mine.shape
assert mine[0].tolist() == [0.0, 4.0, 8.0], "patch (0,0): canais 0,1,2 daquela posicao"
assert mine[3].tolist() == [3.0, 7.0, 11.0], "patch (1,1) da primeira imagem"
assert mine[4].tolist() == [12.0, 16.0, 20.0], "comeco da segunda imagem"
print("OK — patch vectors corretos:", mine.shape)
# (a versao torch equivalente e' PatchFeatureExtractor.to_patch_vectors, usada na Parte 5.)

## Parte 2 — Construir o banco de memória (coreset)

*Leia: `MemoryBank.fit` e a nota do coreset em `src/image_anomaly_lab/detectors/memory_bank.py`.*

Guardar todo patch e' redundante. Fique com uma fracao `ratio` deles, escolhida por
uma RNG **semeada** (para reprodutibilidade), **sem repetir** patch (subconjunto).

In [ ]:
def build_bank(vectors, ratio=0.1, seed=42):
    """Subamostra `vectors` (N, C) para um coreset com round(N*ratio) linhas."""
    # SUA VEZ: use np.random.default_rng(seed).choice(..., replace=False).
    raise NotImplementedError("sua vez!")

In [ ]:
# CHECK — mesmo tamanho e mesma reprodutibilidade do gabarito.
from image_anomaly_lab.config import PatchCoreConfig
from image_anomaly_lab.detectors.memory_bank import MemoryBank

mine = build_bank(good_patches, ratio=0.1, seed=42)
assert mine.shape[0] == round(len(good_patches) * 0.1), mine.shape
again = build_bank(good_patches, ratio=0.1, seed=42)
assert np.array_equal(mine, again), "mesma seed deve dar o mesmo coreset"
print("OK — banco com", mine.shape[0], "patches, reprodutivel")

## Parte 3 — Score = distância ao vizinho mais próximo

*Leia: `MemoryBank.patch_scores` em `src/image_anomaly_lab/detectors/memory_bank.py`.*

Este e' o coração — e o mesmo kNN do seu triplet loss. Para cada patch de consulta,
devolva a distância euclidiana ao patch **mais próximo** do banco. Patches normais
devem pontuar baixo; outliers, alto. Dica: `sklearn.neighbors.NearestNeighbors`.

In [ ]:
def nearest_distance(bank, queries):
    """Para cada linha de `queries` (M, C), distancia ao vizinho mais proximo em `bank`."""
    # SUA VEZ: ajuste um NearestNeighbors(n_neighbors=1) no banco e consulte.
    raise NotImplementedError("sua vez!")

In [ ]:
# CHECK — outliers pontuam mais que inliers, e bate com o gabarito.
bank_ref = MemoryBank(PatchCoreConfig(coreset_ratio=1.0, n_neighbors=1)).fit(good_patches)

d_in = nearest_distance(bank_ref.bank, test_inliers)
d_out = nearest_distance(bank_ref.bank, test_outliers)
assert d_in.max() < d_out.min(), "outliers deveriam ficar todos acima dos inliers"
assert np.allclose(d_in, bank_ref.patch_scores(test_inliers), atol=1e-4)
print(f"OK — inliers ~{d_in.mean():.3f}  vs  outliers ~{d_out.mean():.3f}")

## Parte 4 — Medir a detecção: image AUROC

*Leia: `image_auroc` em `src/image_anomaly_lab/evaluation.py`.*

Dado um score por imagem e o rotulo (0 boa, 1 defeito), o AUROC responde: "para uma
boa e uma defeituosa aleatorias, com que frequencia a defeituosa recebe score
maior?" Implemente (pode usar `sklearn.metrics.roc_auc_score`).

In [ ]:
def image_auroc(labels, scores):
    # SUA VEZ: uma linha com roc_auc_score resolve.
    raise NotImplementedError("sua vez!")

In [ ]:
# CHECK — caso perfeitamente separavel deve dar 1.0, e bater com o gabarito.
from image_anomaly_lab.evaluation import image_auroc as ref_auroc

# score por 'imagem' = maior distancia entre seus patches (aqui, 1 patch por imagem)
labels = np.array([0]*50 + [1]*50)
scores = np.concatenate([d_in, d_out])
assert abs(image_auroc(labels, scores) - 1.0) < 1e-9
assert abs(image_auroc(labels, scores) - ref_auroc(labels, scores)) < 1e-9
print("OK — AUROC =", image_auroc(labels, scores))

## Parte 5 — Juntando em imagens reais (precisa de torch + MVTec)

Agora o pipeline completo com o backbone congelado e o MVTec. Requer:
```bash
pip install --index-url https://download.pytorch.org/whl/rocm6.4 torch torchvision
python scripts/download_data.py --category metal_nut
```
Quase tudo aqui ja vem pronto (o gabarito do `src/`); sobra **uma lacuna**: o
veredito passa/falha. Leia `threshold_youden` em `src/image_anomaly_lab/evaluation.py`.

In [ ]:
from image_anomaly_lab import DataConfig, BackboneConfig, PatchCoreConfig, describe_torch, resolve_torch_device
from image_anomaly_lab.backbones import PatchFeatureExtractor
from image_anomaly_lab.data import build_dataloader
from image_anomaly_lab.detectors.memory_bank import fit_memory_bank, score_split

print(describe_torch().as_dict())
device = resolve_torch_device()
data_cfg = DataConfig(category="metal_nut")

extractor = PatchFeatureExtractor(BackboneConfig(), device)
train_loader = build_dataloader(data_cfg, "train", batch_size=8)
bank, grid_hw = fit_memory_bank(extractor, train_loader, PatchCoreConfig())
print("banco:", bank.bank.shape, "grade de patches:", grid_hw)

In [ ]:
test_loader = build_dataloader(data_cfg, "test", batch_size=8)
res = score_split(extractor, bank, test_loader, data_cfg.image_size)
print("AUROC de imagem:", round(ref_auroc(res["labels"], res["image_scores"]), 4))

In [ ]:
def verdict(image_scores, labels):
    """Escolha um threshold (Youden's J) e devolva um array booleano 'e' defeito?'."""
    # SUA VEZ: use threshold_youden(labels, image_scores) e compare os scores a ele.
    raise NotImplementedError("sua vez!")

In [ ]:
# CHECK — o veredito deve acertar a maioria (metal_nut e' um caso facil).
from image_anomaly_lab.evaluation import threshold_youden

pred = verdict(res["image_scores"], res["labels"])
acc = (pred.astype(int) == res["labels"]).mean()
print(f"acuracia do veredito: {acc:.3f}")
assert acc > 0.85, "algo errado no threshold/comparacao"
print("OK — pipeline completo de ponta a ponta")

## Fim

Você implementou, do zero, as quatro peças do PatchCore-lite (vetores de patch,
banco/coreset, distância kNN, AUROC) e as ligou a imagens reais. Para experimentar
de verdade, vá ao `STUDY_GUIDE.md`: trocar camadas do backbone, encolher o coreset,
e comparar com o autoencoder.